# Sync Preview — Playlist Delta

Shows what a full sync would do **without writing anything**.

- **1 Spotify API call** (fetch playlists)
- Read-only DB connection
- Classifies every playlist as: `new` | `stale` | `current` | `excluded`

Run the *Trigger Sync* section at the bottom only when ready to write.

In [ ]:
import os
import sys
from pathlib import Path

# Find project root by pyproject.toml — works regardless of where the kernel starts
ROOT = next(
    p for p in [Path().resolve(), *Path().resolve().parents] if (p / "pyproject.toml").exists()
)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

from utils.logging import configure_logging

configure_logging()  # must run before any module that calls get_logger()

import polars as pl

from etl.db import get_connection, init_schema
from etl.sync import plan_sync

## Spotify Auth check

In [ ]:
import json as _json
import time

import httpx

from spotify.auth import CACHE_PATH, SpotifyAuth  # absolute path — cwd-independent

if not CACHE_PATH.exists():
    print(f"NO CACHE at {CACHE_PATH}")
    print("Run `make auth` in terminal to authenticate.")
else:
    data = _json.loads(CACHE_PATH.read_text())
    at = data.get("access_token", "")
    exp = data.get("expires_at", 0)
    expired = time.time() > exp - 60
    print(f"cache path  : {CACHE_PATH}")
    print(f"token len   : {len(at)}")
    print(f"expires_at  : {exp:.0f}  ({'EXPIRED' if expired else 'valid'})")
    r = httpx.get("https://api.spotify.com/v1/me", headers={"Authorization": f"Bearer {at}"})
    print(f"GET /v1/me  : {r.status_code}  {r.json().get('display_name', r.text[:80])}")

In [ ]:
# Force-refresh token (uses refresh_token — no browser needed)
# If this raises SpotifyAuthError, run `make auth` in a terminal first.
token = SpotifyAuth().force_refresh()
print(f"New token: {token[:20]}...")

In [ ]:
# Debug: confirm what token SpotifyClient actually sends
from pathlib import Path

from spotify.auth import SpotifyAuth
from utils.config import settings

auth = SpotifyAuth()
token = auth.get_token()

print(f"cache path  : {Path(settings.spotify_cache_path).resolve()}")
print(f"token len   : {len(token)}")
print(f"token prefix: {token[:20]}..." if token else "token: EMPTY ← this is the problem")
print(f"header      : Bearer {token[:20]}...")

## DB state (no API)

In [ ]:
conn = get_connection(read_only=True)

tables = ["playlists", "tracks", "audio_features", "artists", "track_artists", "playlist_tracks"]
for t in tables:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"{t:<20}: {n:,}")

# Tracks missing audio features (sync would need to heal these)
missing_audio = conn.execute("""
    SELECT COUNT(*) FROM tracks t
    LEFT JOIN audio_features af USING (track_id)
    WHERE af.track_id IS NULL
""").fetchone()[0]

missing_artists = conn.execute("""
    SELECT COUNT(DISTINCT ta.artist_id) FROM track_artists ta
    LEFT JOIN artists a USING (artist_id)
    WHERE a.artist_id IS NULL
""").fetchone()[0]

print("\nGaps to heal:")
print(f"  tracks missing audio features : {missing_audio:,}")
print(f"  artists missing data          : {missing_artists:,}")

In [ ]:
# if artist missing data, fetch data
# if audio feature data missing, fetch data
# what about genre?

## Fetch playlists from Spotify API

In [ ]:
import json
import time
from pathlib import Path

from spotify.client import SpotifyClient
from spotify.fetch import fetch_my_playlists

_PLAYLIST_CACHE = Path("/tmp/spotify_playlists_preview.json")
_CACHE_TTL = 3600  # 1 hour


def _cache_fresh() -> bool:
    return _PLAYLIST_CACHE.exists() and (time.time() - _PLAYLIST_CACHE.stat().st_mtime) < _CACHE_TTL


if _cache_fresh():
    raw_playlists = json.loads(_PLAYLIST_CACHE.read_text())
    age = int(time.time() - _PLAYLIST_CACHE.stat().st_mtime)
    print(
        f"Loaded {len(raw_playlists)} playlists from cache (age {age}s — re-run with _PLAYLIST_CACHE.unlink() to force refresh)"
    )
else:
    client = SpotifyClient()
    raw_playlists = fetch_my_playlists(client)
    _PLAYLIST_CACHE.write_text(json.dumps(raw_playlists))
    print(f"Fetched {len(raw_playlists)} playlists from Spotify → cached to {_PLAYLIST_CACHE}")

In [ ]:
raw_playlists

### Stored playlists

In [ ]:
stored_playlists = conn.execute("""
    SELECT *
    FROM playlists
""").pl()
stored_playlists.filter(pl.col("gen_4").is_not_null())[:10]

In [ ]:
stored_playlists.filter(pl.col("gen_4").is_not_null())["playlist_name"].to_list()

### Feature gaps

In [ ]:
# Tracks missing audio features — by playlist
missing_af_by_playlist = conn.execute("""
    SELECT p.playlist_name, COUNT(*) AS missing_audio
    FROM tracks t
    LEFT JOIN audio_features af USING (track_id)
    JOIN playlist_tracks pt USING (track_id)
    JOIN playlists p USING (playlist_id)
    WHERE af.track_id IS NULL
    GROUP BY p.playlist_name
    ORDER BY missing_audio DESC
""").pl()

# Artists missing data — top by track count
missing_artist_detail = conn.execute("""
    SELECT ta.artist_id, COUNT(DISTINCT ta.track_id) AS n_tracks
    FROM track_artists ta
    LEFT JOIN artists a USING (artist_id)
    WHERE a.artist_id IS NULL
    GROUP BY ta.artist_id
    ORDER BY n_tracks DESC
    LIMIT 20
""").pl()

missing_artists = missing_artist_detail["n_tracks"].sum() or 0

print("Tracks missing audio features (by playlist):")
print(missing_af_by_playlist if len(missing_af_by_playlist) else "  None")
print("\nTop artists missing data (showing up to 20):")
print(missing_artist_detail)

## Sync delta — what would change?

In [ ]:
items = plan_sync(conn, raw_playlists)

delta = pl.DataFrame(
    [
        {
            "playlist_name": i.playlist_name,
            #'status': i.status,
            "spotify_tracks": i.spotify_track_count,
            "db_tracks": i.db_track_count,
            "delta": i.spotify_track_count - i.db_track_count,
            "needs_sync": i.needs_sync,
        }
        for i in items
    ]
)

print(delta)

## API call estimate

In [ ]:
to_sync = [i for i in items if i.needs_sync]
total_new_tracks = sum(
    i.spotify_track_count - i.db_track_count
    for i in to_sync
    if i.spotify_track_count > i.db_track_count
)

# Spotify limits: tracks=100/req, audio_features=100/req, artists=50/req
track_fetch_calls = sum(
    max(1, -(-i.spotify_track_count // 100))  # ceiling division
    for i in to_sync
)
audio_calls = max(1, -(-missing_audio // 100)) if missing_audio else 0
artist_calls = max(1, -(-missing_artists // 50)) if missing_artists else 0

print(f"Playlists to sync : {len(to_sync)} / {len(items)}")
print("  status breakdown:")
for status in ["new", "stale", "current", "excluded"]:
    # n = sum(1 for i in items if i.status == status)
    marker = " ← will sync" if status in ("new", "stale") else ""
    print(f"    {status:<10}: {n}{marker}")

print("\nEstimated API calls:")
print("  fetch playlists        : 1 (already done)")
print(f"  fetch tracks           : ~{track_fetch_calls} calls ({len(to_sync)} playlists × pages)")
print(f"  audio features (gaps)  : ~{audio_calls} calls ({missing_audio:,} missing tracks)")
print(f"  artist features (gaps) : ~{artist_calls} calls ({missing_artists:,} missing artists)")
print(f"  TOTAL                  : ~{1 + track_fetch_calls + audio_calls + artist_calls} calls")

## Ignore list — exclude / include playlists

Edit `EXCLUDE_NAME` / `INCLUDE_NAME` below and run the relevant cell. Changes take effect on the next sync.

In [ ]:
# Exclude a playlist — edit name, then run
EXCLUDE_NAME = "Playlist Name"  # ← change this

conn.close()
write_conn = get_connection(read_only=False)
write_conn.execute(
    "UPDATE playlists SET include_in_refresh=FALSE WHERE playlist_name=?",
    [EXCLUDE_NAME],
)
write_conn.close()
print(f"Excluded: {EXCLUDE_NAME}")

In [ ]:
# Re-include a playlist — edit name, then run
INCLUDE_NAME = "Playlist Name"  # ← change this

write_conn = get_connection(read_only=False)
write_conn.execute(
    "UPDATE playlists SET include_in_refresh=TRUE WHERE playlist_name=?",
    [INCLUDE_NAME],
)
write_conn.close()
print(f"Re-included: {INCLUDE_NAME}")

---
## Trigger Sync

Run cells below only when ready to write. Each step is independent.

Set limits in cell 24 for safe testing — set to `None` for a full sync.

| Param | Effect |
|-------|--------|
| `MAX_PLAYLISTS=1` | Only sync tracks for 1 playlist |
| `MAX_TRACKS=5` | Cap total tracks fetched |
| `AUDIO_LIMIT=10` | Cap audio feature fetches |
| `ARTIST_LIMIT=10` | Cap artist feature fetches |

In [ ]:
from etl.sync import sync_artist_features, sync_audio_features, sync_tracks, upsert_playlists

# ── Limits — set to None for a full sync ─────────────────────────────────────
MAX_PLAYLISTS: int | None = None  # cap playlists with track fetches
MAX_TRACKS: int | None = None  # cap total tracks fetched
AUDIO_LIMIT: int | None = None  # cap audio feature fetches
ARTIST_LIMIT: int | None = None  # cap artist feature fetches

conn.close()
write_conn = get_connection(read_only=False)
init_schema(write_conn)
print(
    f"Write connection open. Limits: playlists={MAX_PLAYLISTS}, tracks={MAX_TRACKS}, audio={AUDIO_LIMIT}, artists={ARTIST_LIMIT}"
)

In [ ]:
# Step 1: Register / rename playlists
upsert_playlists(write_conn, items)

In [ ]:
# Step 2: Fetch tracks for new/stale playlists only
all_tracks = sync_tracks(
    write_conn, client, items, max_playlists=MAX_PLAYLISTS, max_tracks=MAX_TRACKS
)

In [ ]:
# Step 3: Fill missing audio features
n_audio = sync_audio_features(write_conn, client, limit=AUDIO_LIMIT)
print(f"Audio features inserted: {n_audio}")

In [ ]:
# Step 4: Fill missing artist data
n_artists = sync_artist_features(write_conn, client, limit=ARTIST_LIMIT)
print(f"Artist rows inserted: {n_artists}")

In [ ]:
# Verify
write_conn.close()
conn = get_connection(read_only=True)

for t in tables:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"{t:<20}: {n:,}")

n_profile = conn.execute("SELECT COUNT(*) FROM track_profile").fetchone()[0]
print(f"\ntrack_profile view       : {n_profile:,} rows")

In [ ]:
# Run this before switching to data_refresh.ipynb — DuckDB allows only one
# connection configuration per file at a time across kernels.
try:
    conn.close()
except Exception:
    pass
try:
    write_conn.close()
except Exception:
    pass
print("All connections closed.")